# Objetivo
Generar samples y obtener una idea del escalado temporal, todo aprovechando de usar tecnicas de batching programadas; de esta manera generando datos para los modelos de ML clasificativos.

In [ ]:
import numpy as np
import pandas as pd
import time
import os
#import psutil
import glob
import pickle
from scipy.stats import qmc
from lib.oracle import OracleExecutor  # assumes your OracleExecutor is in oracle_wrapper.py


In [ ]:

epsilon = 0.001
vev = 246

def generate_local_variations(
    m_phi_base: float,
    m_A_center: float,
    m12_center: float,
    batch_size: int,
    eps_A: float,
    eps_m12: float,
    seed: int = None
    ) -> np.ndarray:
    """
    Generate `batch_size` points where only m_A and m12_2 vary in a small Latin 
    Hypercube around (m_A_center, m12_center), and all other 5 dims are fixed.

    Returns an array of shape (batch_size, 7) with column order:
      [m_phi, m_A, sin_ba, tan_beta, lambda6, lambda7, m12_2]
    """

    # 1) LatinHypercube in 2D (for m_A and m12)
    sampler = qmc.LatinHypercube(d=2)
    unit = sampler.random(n=batch_size)

    # 2) Scale to [m_A_center±eps_A] × [m12_center±eps_m12]
    bounds = np.array([
        [m_A_center - eps_A, m_A_center + eps_A],
        [m12_center - eps_m12, m12_center + eps_m12]
    ])
    scaled = qmc.scale(unit, bounds[:,0], bounds[:,1])  # shape (batch_size,2)

    # 3) Build the full parameter array, hard-coding the other 5 dims:
    P = np.empty((batch_size, 7), dtype=float)
    P[:, 0] = m_phi_base           # m_phi
    P[:, 1] = scaled[:, 0]         # m_A (varying)
    P[:, 2] = 1.0                  # sin(beta - alpha), fixed
    P[:, 3] = 10000.0              # tan(beta), fixed
    P[:, 4] = 0.1                  # lambda_6, fixed
    P[:, 5] = 0.0                  # lambda_7, fixed
    P[:, 6] = scaled[:, 1]         # m12_2 (varying)

    return P


def get_parameters_from_points(
    csv_path: str,
    batch_size: int,
    eps_A: float,
    eps_m12: float,
    seed: int = None
    ) -> np.ndarray:
    """
    Reads `csv_path` containing base points with columns 'Mh2', 'Mh3', 'm12_2' and
    for each row generates `batch_size` local variations via generate_local_variations.
    Returns a combined array of shape (n_rows * batch_size, 7).
    """
    df = pd.read_csv(csv_path)
    all_batches = []
    for idx, row in df.iterrows():
        m_phi_base   = row["Mh2"]    # heavy CP-even Higgs mass as m_phi
        m_A_center   = row["Mh3"]    # CP-odd Higgs mass as m_A
        m12_center   = row["m12_2"]
        # derive unique seed per batch for reproducibility
        batch_seed = None if seed is None else seed + idx
        P = generate_local_variations(
            m_phi_base, m_A_center, m12_center,
            batch_size, eps_A, eps_m12, seed=batch_seed
        )
        all_batches.append(P)
    # stack all batches into one array
    return np.vstack(all_batches)


import numpy as np
from scipy.stats import qmc

def generate_local_variations_phi(
    m_phi_center: float,
    m12_center: float,
    batch_size: int,
    eps_phi: float,
    eps_m12: float,
    m_A_fixed: float = 300.0,
    seed: int = None
) -> np.ndarray:
    """
    Genera `batch_size` puntos variando m_phi y m12_2 en un Latin Hypercube
    alrededor de (m_phi_center, m12_center).
    m_A se fija a `m_A_fixed`. Las otras 4 dimensiones están hardcodeadas.
    Retorna un array de forma (batch_size, 7) con columnas:
    [m_phi, m_A, sin_ba, tan_beta, lambda6, lambda7, m12_2]
    """
    # 1) Muestreo LH en 2D para (m_phi, m12_2)
    sampler = qmc.LatinHypercube(d=2)
    unit = sampler.random(n=batch_size)

    # 2) Escalado a [m_phi_center±eps_phi] × [m12_center±eps_m12]
    bounds = np.array([
        [m_phi_center - eps_phi, m_phi_center + eps_phi],
        [m12_center  - eps_m12,  m12_center  + eps_m12]
    ])
    scaled = qmc.scale(unit, bounds[:,0], bounds[:,1])  # shape (batch_size,2)

    # 3) Construir el array de parámetros
    P = np.empty((batch_size, 7), dtype=float)
    P[:, 0] = scaled[:, 0]       # m_phi (variado)
    P[:, 1] = m_A_fixed           # m_A (fijo)
    P[:, 2] = 1.0                 # sin(beta - alpha)
    P[:, 3] = 10000.0             # tan(beta)
    P[:, 4] = 0.1                 # lambda6
    P[:, 5] = 0.0                 # lambda7
    P[:, 6] = scaled[:, 1]       # m12_2 (variado)

    return P

def get_parameters_from_points_phi(
    csv_path: str,
    batch_size: int,
    eps_phi: float,
    eps_m12: float,
    m_A_fixed: float = 300.0,
    seed: int = None
) -> np.ndarray:
    """
    Lee `csv_path` con columnas 'Mh2' (m_phi_center) y 'm12_2'.
    Para cada fila genera un batch con `generate_local_variations_phi`.
    """
    import pandas as pd
    df = pd.read_csv(csv_path)
    all_batches = []
    for idx, row in df.iterrows():
        m_phi_center = row["Mh2"]
        m12_center   = row["m12_2"]
        batch_seed   = None if seed is None else seed + idx
        P = generate_local_variations_phi(
            m_phi_center,
            m12_center,
            batch_size,
            eps_phi,
            eps_m12,
            m_A_fixed=m_A_fixed,
            seed=batch_seed
        )
        # DEBUG: validación rápida
        # assert np.all(P[:,1] == m_A_fixed), "m_A no está fijo a 300"
        all_batches.append(P)

    return np.vstack(all_batches)

# Ejemplo de uso:
# >>> params = get_parameters_from_points_phi(
#       "puntos_base.csv",
#       batch_size=100,
#       eps_phi=0.5,
#       eps_m12=1.0,
#       m_A_fixed=300.0,
#       seed=42
#     )
# >>> print(params.shape)  # debería ser (n_rows * 100, 7)



# Prepare executor
executor = OracleExecutor(nthreads=4)



In [2]:
import os
import glob
import time
import pickle
import pandas as pd
from typing import Literal
from lib.oracle import OracleExecutor

def multiple_runs(
    csv_path: str,
    N_repeat_runs: int,
    batch_size: int,
    eps_A: float,
    eps_m12: float,
    outdir: str,
    executor: OracleExecutor,
    variation_mode: Literal['mA', 'mPhi'] = 'mA',
    eps_phi: float = None,
    base_seed: int = 42
):
    """
    Ejecuta varias corridas locales LH. Dependiendo de `variation_mode`:
      - 'mA': varía (m_A, m12_2), fija m_phi (usa eps_A)
      - 'mPhi': varía (m_phi, m12_2), fija m_A=300, usa eps_phi
    """
    # Validaciones básicas
    if variation_mode == 'mPhi' and eps_phi is None:
        raise ValueError("Para mode='mPhi' debes proporcionar eps_phi")
    
    df = pd.read_csv(csv_path)
    n_runs = len(df)

    # Estimación de tiempo
    time_per_point = 415.7 / 15_000
    total_points = N_repeat_runs * n_runs * batch_size
    pred_mins = total_points * time_per_point / 60
    print(f"Estimado: {pred_mins:.1f} min (~{pred_mins/60:.2f} h) para {total_points} puntos")

    os.makedirs(outdir, exist_ok=True)

    for epoch in range(N_repeat_runs):
        for j, row in df.iterrows():
            m_phi_base = float(row["Mh2"])
            m_A_center = float(row["Mh3"])
            m12_center = float(row["m12_2"])
            # Semilla reproducible única
            seed = base_seed + epoch * n_runs + j

            # Seleccionar la función de variación según el modo
            if variation_mode == 'mA':
                param_list = generate_local_variations(
                    m_phi_base=m_phi_base,
                    m_A_center=m_A_center,
                    m12_center=m12_center,
                    batch_size=batch_size,
                    eps_A=eps_A,
                    eps_m12=eps_m12,
                    seed=seed
                )
            else:  # 'mPhi'
                param_list = generate_local_variations_phi(
                    m_phi_center=m_phi_base,
                    m12_center=m12_center,
                    batch_size=batch_size,
                    eps_phi=eps_phi,
                    eps_m12=eps_m12,
                    m_A_fixed=m_A_center,  # o un valor fijo, p.ej. 300
                    seed=seed
                )

            # Índice de batch basado en archivos existentes
            existing = sorted(glob.glob(os.path.join(outdir, "batch_*.pkl")))
            batch_idx = len(existing) + 1

            # Ejecución
            t0 = time.perf_counter()
            results = executor.map(param_list.tolist(), use_threads=True)
            dt = time.perf_counter() - t0

            # Guardar resultados
            fname = f"batch_{batch_idx}_{int(m_phi_base)}.pkl"
            fout = os.path.join(outdir, fname)
            with open(fout, "wb") as f:
                pickle.dump({"params": param_list, "results": results}, f)

            print(f"[Run {j+1}/{n_runs}] batch {batch_idx} "
                  f"mode={variation_mode} saved in {dt:.1f}s → {fout}")

        print(f"[Epoch {epoch+1}/{N_repeat_runs}] completado")
    print("Todas las corridas finalizadas.")


# Runs

In [ ]:
import os
import glob
import time
import pickle

import numpy as np
from scipy.stats import qmc
import pandas as pd

# Asegúrate de importar tu executor y multiple_runs
from lib.oracle import OracleExecutor

# ── Parámetros de usuario ─────────────────────────────────────────────────────
CSV_PATH      = "valid_points_lhe/valid_points.csv"
OUTDIR        = "data_batches"
N_RUNS        = 2           # cuántos puntos base correr
BATCH_SIZE    = 3_000       # puntos por run
EPS_A         = 1         # ± variación en m_A (no usado en modo mPhi)
EPS_PHI       = 1         # ± variación en m_phi (nuevo)
EPS_M12       = 2         # ± variación en m12^2
#SEED          = 100         # semilla base

# ── Preparación ────────────────────────────────────────────────────────────────
os.makedirs(OUTDIR, exist_ok=True)
existing_batches = sorted(glob.glob(f"{OUTDIR}/batch_*.pkl"))
print(f"Próximo batch id: {len(existing_batches) + 1}")

executor = OracleExecutor(nthreads=4)

# ── Ejecución ─────────────────────────────────────────────────────────────────
multiple_runs(
    csv_path=CSV_PATH,
    N_repeat_runs=N_RUNS,
    batch_size=BATCH_SIZE,
    eps_A=EPS_A,                   # ignorado en 'mPhi'
    eps_m12=EPS_M12,
    outdir=OUTDIR,
    executor=executor,
    variation_mode='mPhi',         # ¡aquí cambiamos el modo!
    eps_phi=EPS_PHI               # ± rango para m_phi
)

print("Run variando m_phi completada.")


Próximo batch id: 47
Estimado: 63.7 min (~1.06 h) para 138000 puntos
[Run 1/23] batch 47 mode=mPhi saved in 11.5s → data_batches/batch_47_125.pkl
[Run 2/23] batch 48 mode=mPhi saved in 10.1s → data_batches/batch_48_125.pkl
[Run 3/23] batch 49 mode=mPhi saved in 10.4s → data_batches/batch_49_125.pkl
[Run 4/23] batch 50 mode=mPhi saved in 11.4s → data_batches/batch_50_130.pkl
[Run 5/23] batch 51 mode=mPhi saved in 11.3s → data_batches/batch_51_139.pkl
[Run 6/23] batch 52 mode=mPhi saved in 12.7s → data_batches/batch_52_150.pkl
[Run 7/23] batch 53 mode=mPhi saved in 10.9s → data_batches/batch_53_160.pkl
[Run 8/23] batch 54 mode=mPhi saved in 11.1s → data_batches/batch_54_170.pkl
[Run 9/23] batch 55 mode=mPhi saved in 11.6s → data_batches/batch_55_180.pkl
[Run 10/23] batch 56 mode=mPhi saved in 14.4s → data_batches/batch_56_190.pkl
[Run 11/23] batch 57 mode=mPhi saved in 14.2s → data_batches/batch_57_200.pkl


In [19]:
print(param_list[:5])
print(len(param_list))

print(results[:5])
print(len(results))

# all pickles are made like:
#    pickle.dump({"params": param_list, "results": results}, f)


[[ 2.90972554e+02  2.42671492e+02  9.99906968e-01  1.83445228e+03
   3.58254116e-05  9.63008916e-05  5.10090209e+00]
 [ 3.90106060e+02  3.52647880e+02  9.99982419e-01  9.86387544e+03
   6.51931101e-05  9.44609250e-05  3.14377118e+00]
 [ 1.45053133e+02  3.97831745e+02  9.99910765e-01  3.91033007e+03
  -5.33026162e-05  9.01694304e-05  3.64750044e+00]
 [ 2.24301321e+02  3.20172639e+02  9.99942371e-01  2.13994012e+03
  -7.65944688e-05  1.70070492e-05  2.27629995e+00]
 [ 1.57675251e+02  2.88391895e+02  9.99979244e-01  5.63763949e+03
  -8.95021400e-05 -5.67457529e-05  2.66907545e+00]]
15000
[{'positivity_ok': None, 'unitarity_ok': None, 'perturbativity_ok': None, 'w_h2_bb': None, 'w_h2_tautau': None, 'w_h2_uu': None, 'w_h2_du': None, 'w_h2_ln': None, 'w_h2_vv': [None, None, None], 'w_h2_gaga': None, 'w_h2_Zga': None, 'w_h2_gg': None, 'w_h2_hh': None, 'w_total_h2': None, 'w_total_top': None, 'branching_ratio_h2_gaga': None, 'lambda1': None, 'lambda2': None, 'lambda3': None, 'lambda4': None, '

In [20]:
# ------------------------
# Determine next batch index
# ------------------------
existing_batches = sorted(glob.glob(f"{outdir}/batch_*.pkl"))
batch_idx = len(existing_batches) + 1
print(batch_idx)

# ------------------------
# Generate Latin-Hypercube sample
# ------------------------
sampler = qmc.LatinHypercube(d=7, seed=batch_idx)
unit_sample = sampler.random(n=batch_size)
param_list = qmc.scale(unit_sample, param_bounds[:,0], param_bounds[:,1])

# ------------------------
# Run your oracle / executor
# ------------------------
t0 = time.perf_counter()
results = executor.map(param_list.tolist(), use_threads=False)
t1 = time.perf_counter()

# ------------------------
# Save this batch
# ------------------------
outfile = f"{outdir}/batch_{batch_idx}.pkl"
with open(outfile, "wb") as f:
    pickle.dump({"params": param_list, "results": results}, f)

print(f"Batch {batch_idx} saved ({batch_size} points) in {t1-t0:.1f}s → {outfile}")



5
Batch 5 saved (15000 points) in 863.3s → data_batches/batch_5.pkl


In [ ]:
results

In [ ]:
# ------------------------
# Determine next batch index
# ------------------------
existing_batches = sorted(glob.glob(f"{outdir}/batch_*.pkl"))
batch_idx = len(existing_batches) + 1
print(batch_idx)

# ------------------------
# Generate Latin-Hypercube sample
# ------------------------
sampler = qmc.LatinHypercube(d=7, seed=batch_idx)
unit_sample = sampler.random(n=batch_size)
param_list = qmc.scale(unit_sample, param_bounds[:,0], param_bounds[:,1])

# ------------------------
# Run your oracle / executor
# ------------------------
t0 = time.perf_counter()
results = executor.map(param_list.tolist(), use_threads=True)
t1 = time.perf_counter()

# ------------------------
# Save this batch
# ------------------------
outfile = f"{outdir}/batch_{batch_idx}.pkl"
with open(outfile, "wb") as f:
    pickle.dump({"params": param_list, "results": results}, f)

print(f"Batch {batch_idx} saved ({batch_size} points) in {t1-t0:.1f}s → {outfile}")



1



# Testing Speed

In [ ]:
# Define the sampling sizes
sample_sizes = [1, 10, 100, 1_000, 10_000]

for n in sample_sizes:
    # Generate Latin Hypercube samples in [0,1]^7, then scale
    sampler = qmc.LatinHypercube(d=7)
    sample_unit = sampler.random(n)
    param_list = qmc.scale(sample_unit, param_bounds[:,0], param_bounds[:,1])
    
    # Measure memory before run
    process = psutil.Process()
    mem_before = process.memory_info().rss
    
    # Run and time
    t0 = time.perf_counter()
    results = executor.map(param_list.tolist(), use_threads=False)
    t1 = time.perf_counter()
    
    mem_after = process.memory_info().rss
    delta_mem = (mem_after - mem_before) / (1024**2)  # in MB
    
    # Save raw results for this batch
    with open(f"oracle_results_{n}.pkl", "wb") as f:
        pickle.dump(results, f)
    
    # Record performance
    perf_records.append({
        "n_points": n,
        "time_sec": t1 - t0,
        "mem_delta_MB": delta_mem
    })
    print(f"Completed batch {n}: time={t1-t0:.2f}s, memory Δ={delta_mem:.1f}MB")

# Save performance table
df_perf = pd.DataFrame(perf_records)
df_perf.to_csv("performance_scaling.csv", index=False)



# Merging

In [2]:
# ------------------------
# Merge old batches if they exceed size threshold
# ------------------------
def merge_batches(folder, batch_prefix="batch_", merged_prefix="merged_", max_size_mb=30):
    # Count existing merged files to avoid overwrite
    existing_merged = sorted(glob.glob(f"{folder}/{merged_prefix}*.pkl"))
    merge_idx = len(existing_merged) + 1
    
    # Only consider raw batch files
    batch_files = sorted(glob.glob(f"{folder}/{batch_prefix}*.pkl"))
    acc_size = 0
    group = []

    for fp in batch_files:
        fsize = os.path.getsize(fp)
        if (acc_size + fsize) / (1024**2) > max_size_mb and group:
            # Merge current group
            merged_data = []
            for gfp in group:
                with open(gfp, "rb") as gf:
                    merged_data.append(pickle.load(gf))
                os.remove(gfp)
            mout = f"{folder}/{merged_prefix}{merge_idx}.pkl"
            with open(mout, "wb") as mf:
                pickle.dump(merged_data, mf)
            print(f"Merged {len(group)} batches into {mout}")
            merge_idx += 1
            group, acc_size = [], 0

        group.append(fp)
        acc_size += fsize

    # Merge any remaining files
    if group:
        merged_data = []
        for gfp in group:
            with open(gfp, "rb") as gf:
                merged_data.append(pickle.load(gf))
            os.remove(gfp)
        mout = f"{folder}/{merged_prefix}{merge_idx}.pkl"
        with open(mout, "wb") as mf:
            pickle.dump(merged_data, mf)
        print(f"Merged {len(group)} batches into {mout}")
    return merged_data

# Call merge

OUTDIR = "data_batches"
merged_data = merge_batches(OUTDIR)
merged_data

Merged 6 batches into data_batches/merged_14.pkl
Merged 6 batches into data_batches/merged_15.pkl
Merged 6 batches into data_batches/merged_16.pkl
Merged 6 batches into data_batches/merged_17.pkl
Merged 6 batches into data_batches/merged_18.pkl
Merged 6 batches into data_batches/merged_19.pkl
Merged 6 batches into data_batches/merged_20.pkl
Merged 6 batches into data_batches/merged_21.pkl
Merged 6 batches into data_batches/merged_22.pkl
Merged 6 batches into data_batches/merged_23.pkl
Merged 6 batches into data_batches/merged_24.pkl
Merged 6 batches into data_batches/merged_25.pkl
Merged 6 batches into data_batches/merged_26.pkl
Merged 6 batches into data_batches/merged_27.pkl
Merged 6 batches into data_batches/merged_28.pkl
Merged 6 batches into data_batches/merged_29.pkl
Merged 6 batches into data_batches/merged_30.pkl
Merged 6 batches into data_batches/merged_31.pkl
Merged 6 batches into data_batches/merged_32.pkl
Merged 6 batches into data_batches/merged_33.pkl
Merged 6 batches int

[{'params': array([[2.99397717e+02, 3.00000000e+02, 1.00000000e+00, ...,
          1.00000000e-01, 0.00000000e+00, 1.57515539e+00],
         [2.41833407e+02, 3.00000000e+02, 1.00000000e+00, ...,
          1.00000000e-01, 0.00000000e+00, 6.78425602e+00],
         [2.93844341e+02, 3.00000000e+02, 1.00000000e+00, ...,
          1.00000000e-01, 0.00000000e+00, 2.69491422e+00],
         ...,
         [2.53664337e+02, 3.00000000e+02, 1.00000000e+00, ...,
          1.00000000e-01, 0.00000000e+00, 6.71114657e+00],
         [2.85380468e+02, 3.00000000e+02, 1.00000000e+00, ...,
          1.00000000e-01, 0.00000000e+00, 7.64471127e+00],
         [2.87220001e+02, 3.00000000e+02, 1.00000000e+00, ...,
          1.00000000e-01, 0.00000000e+00, 7.69449699e+00]]),
  'results': [{'positivity_ok': None,
    'unitarity_ok': None,
    'perturbativity_ok': None,
    'w_h2_bb': None,
    'w_h2_tautau': None,
    'w_h2_uu': None,
    'w_h2_du': None,
    'w_h2_ln': None,
    'w_h2_vv': [None, None, None],
   